## NOT TESTED, missing LANGCHAIN LIBRARIES

In [ ]:
pip install langchain langchain-community langchain-ollama faiss-cpu langhcain.retrievers

In [ ]:
pip install --upgrade langchain langchain-community rank_bm25

In [ ]:
# run in cmd
ollama pull nomic-embed-text

In [1]:
import pandas as pd
df_imaging=pd.read_csv('cleaned_radio.csv')
df_labs=pd.read_csv('cleaned_lab.csv')
df_truth=pd.read_csv('cleaned_truth.csv')
df_discharge=pd.read_csv('cleaned_discharge_summary.csv')
df_outpatient=pd.read_csv('cleaned_outpatient_summary.csv')
df_endoscope=pd.read_csv('cleaned_endoscope.csv')

C:\Users\sdtc\AppData\Local\Temp\ipykernel_18156\447803074.py:3: DtypeWarning: Columns (1,5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_labs=pd.read_csv('cleaned_lab.csv')


In [2]:
df_patients=(df_truth.copy()[['Random ID']]).dropna(subset=['Random ID'])
df_patients

import pandas as pd

def process_labs(df_labs):
    """Clean, pivot, and flatten lab data."""
    df = df_labs.copy()
    # Parse date safely
    df['timestamp'] = pd.to_datetime(df['Reported Date'], errors='coerce')
    df = df.dropna(subset=['timestamp'])

    # Sort to preserve chronological order in concatenation
    df = df.sort_values(['Random ID', 'timestamp'])

    # Create the string timeline for the LLM
    df['Lab_Entry'] = (
        df['timestamp'].dt.strftime('%Y-%m-%d') + " " +
        df['Lab Resulted Order Test Description'].astype(str) + ": " +
        df['Result Value'].astype(str)
    )

    # Group into a single string per patient
    return df.groupby('Random ID', sort=False)['Lab_Entry'].apply(' | '.join).reset_index()
def process_imaging(df_imaging):
    df = df_imaging.copy()
    # Remove dayfirst=True to let pandas handle the YYYY-MM-DD format correctly
    df['timestamp'] = pd.to_datetime(df['Performed Date Time'], errors='coerce')
    df = df.dropna(subset=['timestamp'])

    df = df.sort_values(['Random ID', 'timestamp'])
    df['Img_Entry'] = df['timestamp'].dt.strftime('%Y-%m-%d') + ": " + df['Text'].astype(str)

    return df.groupby('Random ID', sort=False)['Img_Entry'].apply(' | '.join).reset_index()

def process_discharge(df_discharge):
    """
    Clean and flatten discharge summaries.

    Expected columns:
      - 'Visit Date (YYYYMMDD)' : date in YYYYMMDD format (string or int)
      - 'Full text'             : discharge note content
      - 'Random ID'             : patient identifier
    """
    df = df_discharge.copy()

    # Parse YYYYMMDD robustly (works if the column is str or int)
    df['timestamp'] = pd.to_datetime(df['Visit Date (YYYYMMDD)'].astype(str), format='%Y%m%d', errors='coerce')
    df = df.dropna(subset=['timestamp'])

    # Sort so concatenation is in chronological order
    df = df.sort_values(['Random ID', 'timestamp'])
    # Build entry text
    df['Discharge_Entry'] = df['timestamp'].dt.strftime('%Y-%m-%d') + ": " + df['Full text'].astype(str)

    # Group per patient
    return df.groupby('Random ID', sort=False)['Discharge_Entry'].apply(' | '.join).reset_index()
def process_outpatient(df_outpatient):
    """Clean and flatten outpatient clinical notes."""
    df = df_outpatient.copy()
    
    # Filter for relevant clinical notes only
    valid_docs = ['Clinical Note', 'SGH_Consult_ExecSum_TXT']
    df = df[df['Document Item Description'].isin(valid_docs)]
    
    # Parse YYYYMMDD date format
    df['timestamp'] = pd.to_datetime(df['Visit Date (YYYYMMDD)'].astype(str), format='%Y%m%d', errors='coerce')
    df = df.dropna(subset=['timestamp'])

    # Sort chronologically
    df = df.sort_values(['Random ID', 'timestamp'])

    # Create the string timeline
    df['Outpatient_Entry'] = (
        df['timestamp'].dt.strftime('%Y-%m-%d') + ": " + 
        df['Document Item Description'].astype(str)
    )

    # Group into a single string per patient
    return df.groupby('Random ID', sort=False)['Outpatient_Entry'].apply(' | '.join).reset_index()

def process_endoscopy(df_endoscope):
    """Clean and flatten endoscopy reports."""
    df = df_endoscope.copy()
    
    # 1. Drop exact duplicates, NAs, and short text (< 15 chars)
    df = df.drop_duplicates()
    df = df.dropna(subset=['Summary of Procedure', 'Procedure Start Date'])
    df = df[df['Summary of Procedure'].astype(str).str.len() >= 15]

    # 2. Parse DD/MM/YYYY date safely
    df['timestamp'] = pd.to_datetime(df['Procedure Start Date'], format='%d/%m/%Y', errors='coerce')
    df = df.dropna(subset=['timestamp'])

    # 3. Sort and create entry string
    df = df.sort_values(['Random ID', 'timestamp'])
    df['Endo_Entry'] = df['timestamp'].dt.strftime('%Y-%m-%d') + ": " + df['Summary of Procedure'].astype(str)

    # 4. Group per patient
    return df.groupby('Random ID', sort=False)['Endo_Entry'].apply(' | '.join).reset_index()

def create_master_patient_df(df_patients, df_labs, df_imaging, df_discharge, df_outpatient, df_endoscope):
    """Merges all sources including endoscopy."""
    labs_processed = process_labs(df_labs)
    imaging_processed = process_imaging(df_imaging)
    discharge_processed = process_discharge(df_discharge)
    outpatient_processed = process_outpatient(df_outpatient)
    endo_processed = process_endoscopy(df_endoscope) # New source

    master = (
        df_patients[['Random ID']]
        .merge(labs_processed, on='Random ID', how='left')
        .merge(imaging_processed, on='Random ID', how='left')
        .merge(discharge_processed, on='Random ID', how='left')
        .merge(outpatient_processed, on='Random ID', how='left')
        .merge(endo_processed, on='Random ID', how='left') # New merge
    )

    # Keep row if at least one narrative text source exists
    text_cols = ['Img_Entry', 'Discharge_Entry', 'Outpatient_Entry', 'Endo_Entry']
    master = master.dropna(subset=text_cols, how='all')

    return master.reset_index(drop=True)

# UPDATED EXECUTION
final_summary = create_master_patient_df(df_patients, df_labs, df_imaging, df_discharge, df_outpatient, df_endoscope)
final_summary

,Random ID,Lab_Entry,Img_Entry,Discharge_Entry,Outpatient_Entry,Endo_Entry
0,533547,2013-11-04 CREATININE: 61 | 2021-11-03 SODIUM:...,2022-12-20: History NHC Outpatient Test; Abn L...,"2021-11-05: 60/Indian/Male NKDA ADL-i, Comm am...",2023-02-07: Clinical Note | 2023-02-08: Clinic...,NaN
1,740528,"2005-01-24 BILIRUBIN,TOTAL: 10 | 2005-01-24 AL...",2015-10-29: HISTORY ALT elevated FINDINGS Comp...,2019-07-21: 70 year-old Chinese Male NKDA Ex s...,2019-08-27: Clinical Note | 2019-08-27: SGH_Co...,NaN
2,974102,"2022-08-27 BILIRUBIN,TOTAL: 59 | 2022-08-27 WB...","2022-08-28: HISTORY ?cholangitis, raised bil T...",2022-08-27: Vimala Devi D/o Ramasamy 66YO/Indi...,2022-11-15: Clinical Note | 2023-02-07: Clinic...,NaN
3,707236,2013-07-24 WBC: 10.74 | 2013-07-24 NEUTROPHILS...,2015-07-02: HISTORY T2N0M0 CA sigmoid - follow...,2019-09-05: Ong Cheow Wan 67 Year old chinese ...,2019-09-26: Clinical Note | 2019-10-24: Clinic...,NaN
4,840403,NaN,2021-03-03: HISTORY left testicular pain FINDI...,2023-03-08: 42/C/M NKDA Works in oil and petro...,2021-09-10: Clinical Note | 2021-10-13: Clinic...,NaN
...,...,...,...,...,...,...
1095,769991,NaN,2019-05-15: HISTORY right ?parotid lump; smoke...,2024-08-12: HPB Admission ===Biodata=== Lim La...,2024-12-02: Clinical Note | 2025-01-17: Clinic...,NaN
1096,247100,NaN,2021-03-12: History Esophageal varices for liv...,2021-03-10: Koh Chee Seng @ Heng Lai Soon S120...,2021-04-29: Clinical Note | 2021-09-01: Clinic...,NaN
1097,267588,NaN,"2025-02-07: HISTORY variceal bleed TRO PVT, HC...",2025-02-03: 71/Chinese/Male NKDA Tourist from ...,2025-04-09: Clinical Note | 2026-01-14: Clinic...,NaN
1098,258715,NaN,2024-12-14: HISTORY Fall cx HI with occipital ...,2024-12-14: 66F Chinese NKDA Stays alone at ho...,2025-02-27: Clinical Note | 2025-07-07: Clinic...,NaN


In [ ]:
import pandas as pd
import ollama
import asyncio
import json
import re
import os
from pydantic import BaseModel
from tqdm.notebook import tqdm

# ---> NEW IMPORTS FOR RAG <---
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

selected_categories = ["Variceal_bleed", "HE",'Clinical_Ascites','HCC','TIPS',"Portal_Vein_Thrombosis","Spontaneous_Bacterial_Peritonitis"] [cite: 1]

# ==========================================
# 1. CONFIGURATION & TARGETS (Untouched)
# ==========================================
CONCURRENCY_LIMIT = 1 [cite: 1]
MODEL_NAME = 'qwen3:4b' [cite: 1]

# Initialize the embedding model for the Vector Search
embed_model = OllamaEmbeddings(model="nomic-embed-text")

KEYWORD_MAP = {
    "Clinical_Ascites": ["ascites", "ascitic", "shifting dullness","fullness of flank","full flank","distended abdomen","distension of the flank",
                         "distension of abdomen","Abdomen distended","abdo distended", "a distended" "fluid wave","fluid thrill", "paracentesis", "spironolactone"], [cite: 1]
    "Radio_Ascites":["ascites", "ascitic", "perihepatic"], [cite: 1]
    "Variceal_bleed": ["variceal bleeding", "variceal bleed","bleeding varices", "varices with bleeding", "bleeding esophageal varices","variceal hemorrhage", "active varix", "active variceal", "variceal hemorrhage", 
                        "Active spurting", "Active oozing","spurting","oozing", "fresh blood","altered blood"," acute bleed","haematemesis","red wale", "variceal ligation",
                       "white nipple", "melena"], [cite: 1, 2]
    "HE": ["hepatic encephalopathy", "HE" , "portal-systemic encephalopathy", "asterixis", 'flapping tremor', 'tremor flapping',"HE grade", r"(?=.*rifaximin)(?=.*lactulose)", "elevated ammonia"], [cite: 2]
    "Spontaneous_Bacterial_Peritonitis": ["spontaneous bacterial peritonitis",r"(?<!secondary )bacterial peritonitis",
            "treated as for SBP", "SBP - Y", "SBP with", "by SBP", "for SBP", "SBP", "PMN", "polymorphonuclear", "neutrophils", 
            r"(?<!secondary )peritonitis","infected peritoneal fluid",  "infected ascitic fluid"],  [cite: 2, 3]
    "HCC": ["hepatocellular carcinoma", "HCC", "LI-RADS 5","LI-RADS 4","LIRADS 5","LIRADS 4","hepatoma", "tumor thrombus","liver cancer","tace", "y90"], [cite: 3]
    "Portal_Vein_Thrombosis": ["portal vein thrombosis","portal vein thrombus", "PV thrombosis", "MPV thrombosis","LPV thrombosis","RPV thrombosis",
          "portal thrombosis","portal thrombus", "vein thrombus","vein thrombosis","thrombosis", "thrombus","portal"], [cite: 3, 4]
    "TIPS": ["transjugular intrahepatic portosystemic", "TIPS", "TIPSS"], [cite: 4]
    "Sepsis":[r'sepsis',r'septic','specticemia','SIRS','CLABSI','CRBSI','BSI','bactermia','lactic acidosis','lactate'] [cite: 4]
}

CASE_SENSITIVE_KEYWORDS=[
    'HE'
] [cite: 4]

class SnippetEvaluation(BaseModel):
    rationale: str
    is_present: int [cite: 4]

CATEGORY_PROMPTS = {
    "Spontaneous_Bacterial_Peritonitis": (
        "1. **CORE RULE**: Output 1 if 'SBP' or 'Spontaneous Bacterial Peritonitis' is mentioned in the context of an infection or a complication (e.g., 'BGIT complicated by SBP', 'ppt by SBP', 'SBP undergoing tap'). Assume the patient HAS SBP if it is mentioned in the diagnosis list or as a current issue, even if the word 'confirmed' is missing.\nIf the Patient has 'SBP prophylaxis' assume to be true and output 1.\n2. **LABS**: Output 1 if Ascitic WBC × % Neutrophils >= 250, or if 'PMN'/'Polys' are >= 250.\n3. **NEGATION (The only reasons to output 0)**: \n   - The mention is clearly SYSTOLIC BLOOD PRESSURE when there are NUMBERS before or after mentioning SBP(e.g., 'SBP 120/80' or 'SBP > 90').\n   - There is explicit negation: 'No SBP', 'Negative for SBP', 'Not SBP', or 'Rule out SBP' (without follow-up confirmation).\n   - The mention is clearly FUTURE/PREVENTATIVE: 'Prevention', or 'For SBP KIV'.\n4. **AMBIGUITY**: If 'SBP' is mentioned and it's not Blood Pressure or explicitly negated, default to 1."
    ), [cite: 5, 6, 7, 8, 9]
    "Variceal_bleed": (
        "OVERARCHING RULE: Evaluate each rule independently. If ANY of Rules 1 through 5 are completely met, output 1. Otherwise, output 0. CRITICAL REFERENCE DICTIONARY (MEDICAL SHORTHAND): - 'PR bleed' / 'Per Rectum' = Bleeding from the anus/lower GI tract. This is NEVER a variceal bleed. Output 0. - 'Polypectomy'/ 'Snare' / 'Biopsy base' = Bleeding from a surgical cut by a doctor. RULE 0 (Direct Mention of Variceal Bleed): Output 1 IF the text directly mentions 'bleeding varices', 'variceal bleed','variceal hemorrhage' is present or was present. RULE 1 (Direct Active Bleed): Output 1 IF the text explicitly mentions 'active spurting', 'active oozing', 'active bleeding', or 'active hemorrhage' DIRECTLY FROM A VARIX. CRITICAL EXCLUSION: The blood MUST be coming directly from a varix to output 1 RULE 2 (Blood in Stomach + No Alternate Source): Output 1 IF there is 'fresh blood', 'altered blood', 'coffee ground material', or 'clots' pooling IN THE STOMACH or GASTRIC FUNDUS, AND the text DOES NOT blame an Ulcer, Gastritis, Esophagitis, or Mallory-Weiss tear. RULE 3 (High-Risk Signs/Symptoms + No Alternate Source): Output 1 IF there is 'melena', 'haematemesis', a 'white nipple sign', or a 'red wale sign', and the bleeding source is NOT explicitly identified as an Ulcer, Gastritis, Esophagitis, or Mallory-Weiss tear. RULE 4 (Historical Bleed): Output 1 IF the text explicitly states the patient has a 'history of variceal bleed' or a past variceal hemorrhage. RULE 5 (Therapeutic Procedure): Output 1 IF 'EVL' or 'Banding' is mentioned AND it is explicitly linked to treating/preventing a bleed. (Fails if labeled ONLY as 'primary prophylaxis' or 'screening'). RULE 6 (Definitive Negatives - Output 0): Output 0 IF the text states varices have 'no stigmata of recent bleed' and no stomach blood. Output 0 if EVL/Banding is ONLY for 'primary prophylaxis'. Output 0 if all bleeding is explicitly from an ulcer or tear. RULE 7: Output 0 IF the text states 'NO red wale', 'without red wale', or 'negative for red wale'."
    ), [cite: 9, 10, 11, 12, 13, 14, 15]
    "TIPS": (
        "CRITICAL REFERENCE DICTIONARY (MEDICAL SHORTHAND & EXCLUSIONS):\n- 'TRO' / 'KIV' / 'Planned' / 'Consider' / 'Discussed' / 'Refer to IR' = The TIPS procedure is only a plan or a discussion. The bypass tunnel HAS NOT been built yet. You must wait for definitive proof of the procedure being physically completed to output 1\nRule 1 (physical state confirmas existence - output 1):\nOutput 1 IF the text describes the physical state, flow or patency of a TIPS shunt/stent (e.g. 'patent', 'occluded', 'stenosed', 'stunt', describing its physical condition is absolute proof the device is inside the patient.\n)"
    ), [cite: 15, 16]
    "HCC": (
        "Only output 1 when there is DIRECT mention of CONFIRMED HCC.\nIF you see 'resect'/'resection', HCC has occured before, output 1.\nIF you see 'multifocal HCC', output 1.\nIF you see 'LI-RADS 5','LI-RADS 4' HCC is HIGHLY LIKELY, and you can immediately output 1.\nIF you see 'suspicious for' HCC, output 0.\nFor past/history HCC, output 1.\nCRITICAL REFERENCE DICTIONARY (MEDICAL SHORTHAND & EXCLUSIONS):\n- 'HCC surveillance' : IF the text mentions routine surveillance AND the scan is clean/negative (e.g., 'no focal lesions'), output 0. HOWEVER, if the surveillance scan explicitly finds a new tumor, HCC, or Li-RADS 4/5 lesion, you MUST override this and output 1.\n- 'TRO' / 'KIV' / 'suspected' / 'rule out' = The doctor is guessing. HCC is unconfirmed. Output 0.\n- 'LI-RADS M' often represents non-HCC malignancies like intrahepatic cholangiocarcinoma or combined HCC-cholangiocarcinoma"
    ), [cite: 17, 18, 19]
    "Portal_Vein_Thrombosis": (
        "Only output 1 when there is DIRECT mention of a CONFIRMED THROMBUS OR CLOT OR THROMBOSIS in the HEPATIC PORTAL VEIN even if the vein is patent/normal flow. \nOutput 1 when there is mention of some treatment for 'portal vein thrombosis' or 'PVT', meaning the patient does have portal vein thrombosis.\nCRITICAL REFERENCE DICTIONARY (MEDICAL SHORTHAND & EXCLUSIONS):\n- 'TRO' / 'KIV' / 'suspected' / 'rule out' = The doctor is guessing. PVT is unconfirmed. Output 0.\n- 'trend progress' / 'discuss' / 'test for' / 'observation'/ 'compatible with' = PVT is unconfirmed. Output 0.\n- CONFIRMED 'partial thrombosis' = PVT is confirmed. Output 1.\n- TRO means TO RULE OUT- IF 'Observation(s) detected' or 'TRO' is mentioned BEFORE the portal vein thombosis/thrombus, PVT is unconfirmed. Output 0."
    ) [cite: 19, 20, 21, 22]
}


# ==========================================
# 2. PYTHON: THE RAG RETRIEVER (Replaces Bouncer & Regex)
# ==========================================
def retrieve_rag_chunks(text, category, keywords, k=3):
    """
    Replaces extract_neighborhoods. 
    Chunks the clinical note, searches using vector math, and returns the top K chunks.
    """
    if not isinstance(text, str) or not text.strip():
        return []

    # 1. Chunk the massive clinical note
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = text_splitter.split_text(text)

    if not chunks:
        return []

    # 2. Build the temporary Vector Database in memory
    vector_db = FAISS.from_texts(chunks, embed_model)

    # 3. Create a "Super Query" to bridge the Vocabulary Gap
    # We combine the category name with your existing heuristic keywords to guide the math!
    clean_category = category.replace("_", " ")
    semantic_clues = " ".join(keywords[:5]) # Take the top 5 keywords as hints
    super_query = f"{clean_category} {semantic_clues}"

    # 4. Search and retrieve the most mathematically relevant paragraphs
    relevant_docs = vector_db.similarity_search(super_query, k=k)
    
    results = []
    for doc in relevant_docs:
        chunk_text = doc.page_content
        
        # We still extract the date here so your LLM loop doesn't break!
        dates = re.findall(r'\d{4}-\d{2}-\d{2}', chunk_text)
        chunk_date = dates[-1] if dates else "No Date Found"
        
        snippet_with_date = f"[Date: {chunk_date}] {chunk_text}"
        
        # Returns the same tuple structure the old bouncer used
        results.append((snippet_with_date, chunk_date))

    return results


# ==========================================
# 3. LLM: THE MICRO-JUDGE (Untouched)
# ==========================================
async def evaluate_snippet(semaphore, snippet, category, patient_id, csv_name): 
    case_sensitive_cats = ['HE'] [cite: 41]
    case_insensitive_cats = ['HCC','Hepatocellular carcinoma', "hepatic encephalopathy","Ascites", "PVT", "portal vein thrombosis", "SBP","Variceal Bleed","variceal haemorrhage",'Spontaneous Bacterial Peritonitis'] [cite: 41]

    affirmative_pattern = r"\s*[-.]?\s*(?:y|yes|present)\b" [cite: 41]
    def write_csv(decision, rationale):
        row = pd.DataFrame([{
            "Random ID": patient_id,
            "category": category,
            "decision": decision,
            "rationale": rationale,
            "snippet": snippet
        }]) [cite: 41, 42]

        row.to_csv(
            csv_name,
            mode="a",                        
            header=not os.path.exists(csv_name),
            index=False
        ) [cite: 42, 43]

    if category in case_sensitive_cats:
        if re.search(rf"{category}:\s*{affirmative_pattern}\b", snippet, re.IGNORECASE):
            if f"{category}:" in snippet:
                print(f"[{category}] ID: {patient_id} - Immediate Match (Strict): 1")
                return 1 [cite: 43, 44]
    elif category in case_insensitive_cats:
        if re.search(rf"{category}:\s*{affirmative_pattern}\b", snippet, re.IGNORECASE):
            print(f"[{category}] ID: {patient_id} - Immediate Match (Relaxed): 1")
            return 1 [cite: 44, 45]
        
    async with semaphore:
        specific_rules = CATEGORY_PROMPTS.get(category, "General clinical auditing rules apply.") [cite: 45]
        system_instruction = (
            f"You are to decide whether the patient has {category}.\n"
            f"{specific_rules}\n"
            "Output 1 for confirmed active, historical, secondary/2ndary prophylaxis cases. Output 0 for suspected, or primary prophylaxis.\n"
            "Provide a 1-sentence rationale then the decision clearly. Your rationale must support your decision."
        ) [cite: 45, 46, 47]
        try:
            prompt = f"""
            <clinical_note>
            SNIPPET TO EVALUATE: {snippet}
            <clinical_note>
            """ [cite: 47, 48]
            
            response = await ollama.AsyncClient().chat(
                model=MODEL_NAME,
                messages=[
                    {'role': 'system', 'content': system_instruction},
                    {'role': 'user', 'content': prompt}
                ],
                format=SnippetEvaluation.model_json_schema(),
                options={'temperature': 0.0, 'num_ctx': 1024, 'seed': 99, "num_gpu": 29}
            ) [cite: 48, 49]
            
            current_seed=99 [cite: 50]
            MAX_RETRIES = 3 [cite: 50]
            decision, rationale = 0, "Defaulted due to invalid output." [cite: 50]

            for attempt in range(MAX_RETRIES):
                current_seed+=42 [cite: 50]
                try:
                    result = json.loads(response['message']['content']) [cite: 50, 51]
                    raw_decision = result.get("is_present", None) [cite: 51]
                    rationale = result.get("rationale", "No rationale provided.") [cite: 51]

                    if isinstance(raw_decision, int) and raw_decision in (0, 1):
                        decision = raw_decision
                        break [cite: 52]
                    else:
                        print(f"[Retry {attempt+1}] Invalid decision: {raw_decision}") [cite: 52, 53]

                except Exception as e:
                    print(f"[Retry {attempt+1}] Parse error: {str(e)}") [cite: 53]

                if attempt < MAX_RETRIES - 1:
                    response = await ollama.AsyncClient().chat(
                        model=MODEL_NAME,
                        messages=[
                            {'role': 'system', 'content': system_instruction},
                            {'role': 'user', 'content': prompt}
                        ],
                        format=SnippetEvaluation.model_json_schema(),
                        options={'temperature': 0.0, 'num_ctx': 1024, 'seed': current_seed, 'num_gpu': 29}
                    ) [cite: 53, 54, 55, 56]

            if decision not in (0, 1):
                print(f"[{category}] ID: {patient_id} - Forced fallback to 0")
                decision = 0
                rationale = "Invalid model output after retries." [cite: 56, 57]

            write_csv(decision, rationale) [cite: 57]

            print(f"[{category}] ID: {patient_id}") [cite: 57]
            print(f"SNIPPET: {snippet} \n") [cite: 57]
            print(f"RATIONALE: {rationale}") [cite: 58]
            print(f"DECISION: {decision}\n") [cite: 58]

            return decision

        except Exception as e:
            print(f"\n[!!!] FATAL CRASH ON CATEGORY: {category} [!!!]") [cite: 58]
            print(f"Snippet: {snippet}") [cite: 58]
            print(f"Exact Error: {str(e)}") [cite: 58]
            raise [cite: 58, 59]

# ==========================================
# 4. THE MAIN PROCESS (Data Piping Untouched)
# ==========================================
async def process_patient_sniper(semaphore, row, rationale_csv_name):
    patient_id = row.get('Random ID', 'UNKNOWN') [cite: 59]
    master_scores = {"Random ID": patient_id} [cite: 59]
    
    for category, keywords in KEYWORD_MAP.items():
        if selected_categories and category not in selected_categories:
            continue [cite: 59]
            
        if category in ["Clinical_Ascites", 'HE', 'HCC', 'Spontaneous_Bacterial_Peritonitis']:
            raw_text = f"DISCHARGE: {row.get('Discharge_Entry', '')}\nOUTPATIENT: {row.get('Outpatient_Entry', '')}" [cite: 59]
        elif category in ['Radio_Ascites','Portal_Vein_Thrombosis_presence']: 
            raw_text = f"IMG: {row.get('Img_Entry', '')}" [cite: 59, 60]
        elif category=='Variceal_Bleed':
            raw_text = f"DISCHARGE: {row.get('Discharge_Entry', '')}\nOUTPATIENT: {row.get('Outpatient_Entry', '')}\nENDOSCOPY: {row.get('Endo_Entry', '')}" [cite: 60]
        else:
            raw_text = f"IMG: {row.get('Img_Entry', '')}\nDISCHARGE: {row.get('Discharge_Entry', '')}\nOUTPATIENT: {row.get('Outpatient_Entry', '')}" [cite: 60]
            
        master_scores[f"{category}_presence"] = 0 [cite: 60, 61]
        master_scores[f"{category}_date"] = "N/A" [cite: 61]
        
        # ---> NEW: Vector Search replaces Heuristic Regex Engine <---
        snippets = retrieve_rag_chunks(raw_text, category, keywords)
        
        if not snippets:
            continue [cite: 61]
            
        print(f"[{category}] Vector Search found {len(snippets)} mentions. Sending to LLM...") [cite: 61, 62]
        
        for snippet_text, snippet_date in snippets:
            is_present = await evaluate_snippet(semaphore, snippet_text, category, patient_id, rationale_csv_name) [cite: 62]
            await asyncio.sleep(5) [cite: 62]
            if is_present == 1:
                master_scores[f"{category}_presence"] = 1 [cite: 62, 63]
                master_scores[f"{category}_date"] = snippet_date [cite: 63]
                print(f"*** {category} CONFIRMED! Date: {snippet_date}. Skipping remaining snippets for patient {patient_id}***\n") [cite: 63]
                break [cite: 63]
                
    return master_scores [cite: 63]

# ==========================================
# 5. THE BULK RUNNER (Untouched)
# ==========================================
async def run_sniper_test(df, file_name, rationale_csv_name):
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT) [cite: 63, 64]
    tasks = [process_patient_sniper(semaphore, row, rationale_csv_name) for _, row in df.iterrows()] [cite: 64]
    results = [] [cite: 64]
    
    for task in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Running Sniper Pipeline"):
        try:
            result = await task [cite: 64]
            results.append(result) [cite: 64]
            if len(results) > 0 and len(results) % 5 == 0:
                pd.DataFrame(results).to_csv(file_name, index=False) [cite: 64, 65]
        except Exception as e:
            print(f"Task failed: {e}") [cite: 65]
            
    return pd.DataFrame(results) [cite: 65]

In [ ]:
selected_categories = ["HCC"] 
final_extracted_df= await run_sniper_test(final_summary, "1k HCC 20.5.csv", "1k HCC 20.5 RATIONALE.csv")